In [ ]:
import matplotlib.pyplot as plt
import math
import numpy as np
from scipy.integrate import solve_bvp

# physical constants
water_specific_heat = 4184.0
DENSITY_WATER_KG_M3 = 1000.0 #kg/m^3
DENSITY_WATER_KG_L = 1.0

U_PRESETS = {
    "water_convection_moderate": 500,
}

# functions to calculate surface area, cross section, volume
def cylinder_lateral_area(radius_m, height_m):
    return 2 * math.pi * radius_m * height_m
def cylinder_cross_section(radius_m):
    return math.pi * radius_m**2
def cylinder_volume(radius_m, height_m):
    return math.pi * radius_m**2 * height_m
def calculate_counterflow_losses(
    T_supply_c,
    T_out_meas_c,
    mdot_cool_kg_s,
    cp_cool_j_kgk,
    T_hot_in_c,
    mdot_hot_kg_s,
    cp_hot_j_kgk,
    wall_height_m,
    wall_length_m,
    wall_thick_m,
    k_wall_w_mk,
    h_conv_w_m2k,
):

    C_cool = mdot_cool_kg_s * cp_cool_j_kgk
    C_hot = mdot_hot_kg_s * cp_hot_j_kgk

    U_eff = 1.0 / (2.0 / h_conv_w_m2k + wall_thick_m / k_wall_w_mk)

    UA_ch = U_eff * wall_height_m
    UA_oh = U_eff * wall_height_m
    UA_cw = U_eff * wall_height_m

# differential equation definitions (for counterflow)
    def ode(x, T):
        Tc = T[0]
        Th = T[1]
        To = T[2]

        q_ch = UA_ch * (Th - Tc)
        q_oh = UA_oh * (Th - To)
        q_cw = UA_cw * (To - Tc)

        dTc = (q_ch + q_cw) / C_cool
        dTh = (q_ch + q_oh) / C_hot
        dTo = -(q_oh - q_cw) / C_cool

        return np.vstack((dTc, dTh, dTo))

    def bc(T_left, T_right):
        return np.array([
            T_left[0] - T_supply_c,
            T_left[2] - T_out_meas_c,
            T_right[1] - T_hot_in_c,
        ])

    x_guess = np.linspace(0, wall_length_m, 200)
    Tc_guess = np.linspace(T_supply_c, T_supply_c + 2.0, x_guess.size)
    Th_guess = np.linspace(T_hot_in_c - 2.0, T_hot_in_c, x_guess.size)
    To_guess = np.linspace(T_out_meas_c, T_out_meas_c - 2.0, x_guess.size)

    y_guess = np.vstack((Tc_guess, Th_guess, To_guess))
    sol = solve_bvp(ode, bc, x_guess, y_guess, max_nodes=10000)

    if not sol.success:
        raise ValueError(f"Convergence error: {sol.message}")

    x = np.linspace(0, wall_length_m, 1000)
    Tc, Th, To = sol.sol(x)

    q_hot_per_m = UA_ch * (Th - Tc)
    q_warm_per_m = UA_cw * (To - Tc)

    Q_lost_to_hot_W = np.trapz(q_hot_per_m, x)
    Q_lost_to_warm_W = np.trapz(q_warm_per_m, x)
    Q_lost_total_W = Q_lost_to_hot_W + Q_lost_to_warm_W

    return {
        "T_cold_supply_c": T_supply_c,
        "T_cold_distal_c": Tc[-1],
        "T_hot_proximal_c": Th[0],
        "T_hot_distal_c": Th[-1],
        "T_warm_proximal_c": To[0],
        "T_warm_distal_c": To[-1],
        "T_return_used_c": T_out_meas_c,
        "C_cool_W_K": C_cool,
        "C_hot_W_K": C_hot,
        "U_eff_W_m2K": U_eff,
        "UA_ch_W_mK": UA_ch,
        "UA_oh_W_mK": UA_oh,
        "UA_cw_W_mK": UA_cw,
        "wall_length_m": wall_length_m,
        "x_m": x,
        "Tc_c": Tc,
        "Th_c": Th,
        "To_c": To,
        "q_hot_per_m_W_m": q_hot_per_m,
        "q_warm_per_m_W_m": q_warm_per_m,
        "Q_lost_to_hot_W": Q_lost_to_hot_W,
        "Q_lost_to_warm_W": Q_lost_to_warm_W,
        "Q_lost_total_W": Q_lost_total_W,
    }

# calculate cooling power using geometry and counterflow (above)
def calculate_cooling_power(
    cooled_mass_kg,
    total_mass_kg,
    initial_temp_c,
    target_temp_c,
    time_seconds,
    inner_radius_m,
    inner_height_m,
    outer_radius_m,
    outer_height_m,
    u_coeff_w_m2_c,
    outer_temp_c,
    flow_rate_ml_s,
    cop=1.0,
    counterflow_params=None,
    counterflow_max_iter=50,
    counterflow_tol_c=1e-4,
):
    if total_mass_kg is None:
        total_mass_kg = cooled_mass_kg
    if outer_temp_c is None:
        outer_temp_c = initial_temp_c

    a_lateral_inner = cylinder_lateral_area(inner_radius_m, inner_height_m)
    a_cross_inner = cylinder_cross_section(inner_radius_m)
    vol_inner_m3 = cylinder_volume(inner_radius_m, inner_height_m)
    vol_inner_L = vol_inner_m3 * 1000

    a_lateral_outer = cylinder_lateral_area(outer_radius_m, outer_height_m)
    a_cross_outer = cylinder_cross_section(outer_radius_m)
    vol_outer_m3 = cylinder_volume(outer_radius_m, outer_height_m)
    vol_annulus_m3 = vol_outer_m3 - vol_inner_m3

    delta_t_cool = initial_temp_c - target_temp_c
    t_inner_mean = (initial_temp_c + target_temp_c) / 2.0
    delta_t_wall = outer_temp_c - t_inner_mean # uses mean temp to get sense of average power
    delta_t_wall_max = outer_temp_c - target_temp_c # uses max temp to get sense of max power to wall (conservative estimate for losses to body at end of ramp)

    flow_rate_m3_s = flow_rate_ml_s / 1e6
    flow_rate_ml_min = flow_rate_ml_s * 60
    flow_rate_L_min = flow_rate_ml_s * 60 / 1000
    mdot_kg_s = flow_rate_m3_s * DENSITY_WATER_KG_M3
    flow_velocity_m_s = (flow_rate_m3_s / a_cross_inner) if a_cross_inner > 0 else 0.0
    residence_time_s = (vol_inner_m3 / flow_rate_m3_s) if flow_rate_ml_s > 0 else float("inf")

    q_sensible = cooled_mass_kg * water_specific_heat * delta_t_cool
    p_sensible = q_sensible / time_seconds

    p_wall = u_coeff_w_m2_c * a_lateral_inner * delta_t_wall
    p_wall_max = u_coeff_w_m2_c * a_lateral_inner * delta_t_wall_max
    q_wall = p_wall * time_seconds

    p_flow = mdot_kg_s * water_specific_heat * delta_t_cool
    q_flow = p_flow * time_seconds

    q_base = q_sensible + q_wall + q_flow
    thermal_power_base_w = q_base / time_seconds

    counterflow_result = None
    p_counterflow = 0.0
    q_counterflow = 0.0
    return_temp_c = None
    counterflow_iterations = 0

    if counterflow_params is not None:
        cf_params = dict(counterflow_params)

        if cf_params.get("T_out_meas_c", None) is not None:
            return_temp_c = cf_params["T_out_meas_c"]
            counterflow_result = calculate_counterflow_losses(**cf_params)
            p_counterflow = counterflow_result["Q_lost_total_W"]
            q_counterflow = p_counterflow * time_seconds

        else:
            mdot_cool_cf = cf_params["mdot_cool_kg_s"]
            cp_cool_cf = cf_params["cp_cool_j_kgk"]
            T_supply_cf = cf_params["T_supply_c"]
            return_temp_c = T_supply_cf + thermal_power_base_w / (mdot_cool_cf * cp_cool_cf)

            for i in range(counterflow_max_iter):
                cf_params["T_out_meas_c"] = return_temp_c
                counterflow_result = calculate_counterflow_losses(**cf_params)
                p_counterflow = counterflow_result["Q_lost_total_W"]
                q_counterflow = p_counterflow * time_seconds

                thermal_power_candidate_w = (q_base + q_counterflow) / time_seconds
                return_temp_new_c = T_supply_cf + thermal_power_candidate_w / (mdot_cool_cf * cp_cool_cf)

                counterflow_iterations = i + 1

                if abs(return_temp_new_c - return_temp_c) < counterflow_tol_c:
                    return_temp_c = return_temp_new_c
                    cf_params["T_out_meas_c"] = return_temp_c
                    counterflow_result = calculate_counterflow_losses(**cf_params)
                    p_counterflow = counterflow_result["Q_lost_total_W"]
                    q_counterflow = p_counterflow * time_seconds
                    break

                return_temp_c = return_temp_new_c
            else:
                raise ValueError("Convergence error.")

    q_total = q_sensible + q_wall + q_flow + q_counterflow
    thermal_power_w = q_total / time_seconds
    conservative_endpoint_power_w = p_sensible + p_wall_max + p_flow + p_counterflow

    input_power_w = thermal_power_w / cop
    input_power_endpoint_w = conservative_endpoint_power_w / cop

    def pct(q):
        return (q / q_total * 100) if q_total > 0 else 0.0

    return {
        "cooled_mass_kg": cooled_mass_kg,
        "total_mass_kg": total_mass_kg,
        "uncooled_mass_kg": total_mass_kg - cooled_mass_kg,
        "cooled_fraction_pct": cooled_mass_kg / total_mass_kg * 100,
        "initial_temp_c": initial_temp_c,
        "target_temp_c": target_temp_c,
        "outer_temp_c": outer_temp_c,
        "delta_t_cool_c": delta_t_cool,
        "t_inner_mean_c": t_inner_mean,
        "delta_t_wall_c": delta_t_wall,
        "delta_t_wall_max_c": delta_t_wall_max,
        "time_seconds": time_seconds,
        "time_minutes": time_seconds / 60,
        "time_hours": time_seconds / 3600,
        "inner_radius_m": inner_radius_m,
        "inner_radius_mm": inner_radius_m * 1000,
        "inner_height_m": inner_height_m,
        "inner_height_mm": inner_height_m * 1000,
        "a_lateral_inner_m2": a_lateral_inner,
        "a_cross_inner_m2": a_cross_inner,
        "vol_inner_m3": vol_inner_m3,
        "vol_inner_L": vol_inner_L,
        "outer_radius_m": outer_radius_m,
        "outer_radius_mm": outer_radius_m * 1000,
        "outer_height_m": outer_height_m,
        "outer_height_mm": outer_height_m * 1000,
        "a_lateral_outer_m2": a_lateral_outer,
        "a_cross_outer_m2": a_cross_outer,
        "vol_outer_m3": vol_outer_m3,
        "vol_annulus_m3": vol_annulus_m3,
        "u_coeff_w_m2_c": u_coeff_w_m2_c,
        "p_wall_W": p_wall,
        "p_wall_max_W": p_wall_max,
        "q_wall_J": q_wall,
        "q_wall_kJ": q_wall / 1000,
        "q_wall_pct": pct(q_wall),
        "flow_rate_ml_s": flow_rate_ml_s,
        "flow_rate_ml_min": flow_rate_ml_s * 60,
        "flow_rate_L_min": flow_rate_ml_s * 60 / 1000,
        "flow_rate_m3_s": flow_rate_m3_s,
        "mdot_kg_s": mdot_kg_s,
        "flow_velocity_m_s": flow_velocity_m_s,
        "residence_time_s": residence_time_s,
        "total_flow_volume_L": flow_rate_ml_s * time_seconds / 1000,
        "total_flow_mass_kg": mdot_kg_s * time_seconds,
        "p_flow_W": p_flow,
        "q_flow_J": q_flow,
        "q_flow_kJ": q_flow / 1000,
        "q_flow_pct": pct(q_flow),
        "p_sensible_W": p_sensible,
        "q_sensible_J": q_sensible,
        "q_sensible_kJ": q_sensible / 1000,
        "q_sensible_pct": pct(q_sensible),
        "p_counterflow_W": p_counterflow,
        "q_counterflow_J": q_counterflow,
        "q_counterflow_kJ": q_counterflow / 1000,
        "q_counterflow_pct": pct(q_counterflow),
        "counterflow_result": counterflow_result,
        "counterflow_return_temp_c": return_temp_c,
        "counterflow_iterations": counterflow_iterations,
        "q_total_J": q_total,
        "q_total_kJ": q_total / 1000,
        "q_total_kWh": q_total / 3_600_000,
        "thermal_power_W": thermal_power_w,
        "thermal_power_kW": thermal_power_w / 1000,
        "conservative_endpoint_power_W": conservative_endpoint_power_w,
        "conservative_endpoint_power_kW": conservative_endpoint_power_w / 1000,
        "cop": cop,
        "input_power_W": input_power_w,
        "input_power_kW": input_power_w / 1000,
        "input_power_endpoint_W": input_power_endpoint_w,
        "input_power_endpoint_kW": input_power_endpoint_w / 1000,
    }


def print_report(r):
    W = 62
    print("=" * W)
    print("         COOLING POWER CALCULATION RESULTS")
    print("=" * W)

    print("\n  [ Cylinder Geometry ]")
    print(f"  Inner cylinder (cooled zone)")
    print(f"    Radius                   : {r['inner_radius_m']*100:.1f} cm  ({r['inner_radius_mm']:.1f} mm)")
    print(f"    Height                   : {r['inner_height_m']*100:.1f} cm  ({r['inner_height_mm']:.1f} mm)")
    print(f"    Lateral surface area     : {r['a_lateral_inner_m2']:.5f} m²  ({r['a_lateral_inner_m2']*1e4:.2f} cm²)")
    print(f"    Cross-sectional area     : {r['a_cross_inner_m2']:.6f} m²  ({r['a_cross_inner_m2']*1e4:.3f} cm²)")
    print(f"    Volume                   : {r['vol_inner_m3']*1e6:.2f} mL  ({r['vol_inner_L']:.4f} L)")
    print(f"  Outer cylinder (total mass)")
    print(f"    Radius                   : {r['outer_radius_m']*100:.1f} cm  ({r['outer_radius_mm']:.1f} mm)")
    print(f"    Height                   : {r['outer_height_m']*100:.1f} cm  ({r['outer_height_mm']:.1f} mm)")
    print(f"    Annular volume           : {r['vol_annulus_m3']*1e6:.1f} mL")

    print("\n  [ Mass Configuration ]")
    print(f"  Total heat-producing mass  : {r['total_mass_kg']:.3f} kg")
    print(f"  Cooled zone mass           : {r['cooled_mass_kg']:.3f} kg  ({r['cooled_fraction_pct']:.1f}%)")
    print(f"  Uncooled (passive) mass    : {r['uncooled_mass_kg']:.3f} kg  ({100-r['cooled_fraction_pct']:.1f}%)")

    print("\n  [ Thermal Conditions ]")
    print(f"  Initial (inner) temp       : {r['initial_temp_c']:.2f} °C")
    print(f"  Target  (inner) temp       : {r['target_temp_c']:.2f} °C")
    print(f"  Mean inner temp            : {r['t_inner_mean_c']:.2f} °C")
    print(f"  Outer (hot) mass temp      : {r['outer_temp_c']:.2f} °C")
    print(f"  ΔT cooling (inner)         : {r['delta_t_cool_c']:.2f} °C")
    print(f"  ΔT wall (outer−inner mean) : {r['delta_t_wall_c']:.2f} °C")
    print(f"  ΔT wall (endpoint max)     : {r['delta_t_wall_max_c']:.2f} °C")
    print(f"  Cooling time               : {r['time_seconds']:.1f} s  ({r['time_minutes']:.2f} min / {r['time_hours']:.4f} h)")
    print(f"  System COP                 : {r['cop']:.2f}")

    print("\n  [ Radial Wall Heat Transfer ]")
    print(f"  U coefficient              : {r['u_coeff_w_m2_c']:.1f} W/m²·°C")
    print(f"  Wall area (inner lateral)  : {r['a_lateral_inner_m2']:.5f} m²")
    print(f"  Continuous wall load       : {r['p_wall_W']:.3f} W")
    print(f"  Peak wall load             : {r['p_wall_max_W']:.3f} W")

    print("\n  [ Axial Fluid Flow ]")
    if r["flow_rate_ml_s"] > 0:
        print(f"  Flow rate                  : {r['flow_rate_ml_s']:.3f} mL/s  ({r['flow_rate_ml_min']:.2f} mL/min)")
        print(f"  Mass flow rate             : {r['mdot_kg_s']:.6f} kg/s")
        print(f"  Flow velocity (inner bore) : {r['flow_velocity_m_s']:.5f} m/s  ({r['flow_velocity_m_s']*100:.4f} cm/s)")
        print(f"  Residence time in cylinder : {r['residence_time_s']:.1f} s")
        print(f"  Total fluid exchanged      : {r['total_flow_volume_L']:.3f} L  ({r['total_flow_mass_kg']:.3f} kg)")
        print(f"  Continuous flow load       : {r['p_flow_W']:.3f} W")
    else:
        print("  No axial flow specified.")

    if r["counterflow_result"] is not None:
        c = r["counterflow_result"]
        print("\n  [ Counterflow Line Losses ]")
        print(f"  Return temperature used   : {r['counterflow_return_temp_c']:.3f} °C")
        print(f"  Effective U               : {c['U_eff_W_m2K']:.1f} W/m²·K")
        print(f"  UA cold↔hot per m         : {c['UA_ch_W_mK']:.4f} W/(m·K)")
        print(f"  UA hot↔warm per m         : {c['UA_oh_W_mK']:.4f} W/(m·K)")
        print(f"  UA cold↔warm per m        : {c['UA_cw_W_mK']:.4f} W/(m·K)")
        print(f"  Cooling lost to hot line  : {c['Q_lost_to_hot_W']:.3f} W")
        print(f"  Cooling lost to warm line : {c['Q_lost_to_warm_W']:.3f} W")
        print(f"  Total line-loss load      : {c['Q_lost_total_W']:.3f} W")

    print("\n  [ Energy Breakdown ]")
    print(f"  Sensible heat (transient)  : {r['q_sensible_kJ']:>10,.3f} kJ  ({r['q_sensible_pct']:.1f}%)  [{r['p_sensible_W']:.2f} W avg]")
    print(f"  Radial wall transfer       : {r['q_wall_kJ']:>10,.3f} kJ  ({r['q_wall_pct']:.1f}%)  [{r['p_wall_W']:.2f} W]")
    print(f"  Axial fluid inflow         : {r['q_flow_kJ']:>10,.3f} kJ  ({r['q_flow_pct']:.1f}%)  [{r['p_flow_W']:.2f} W]")
    if r["counterflow_result"] is not None:
        print(f"  Counterflow line losses    : {r['q_counterflow_kJ']:>10,.3f} kJ  ({r['q_counterflow_pct']:.1f}%)  [{r['p_counterflow_W']:.2f} W]")
    print(f"  {'─'*52}")
    
    print("\n  [ Power Required ]")
    print(f"  Thermal cooling load       : {r['thermal_power_W']:>10,.3f} W  ({r['thermal_power_kW']:.5f} kW)")
    print(f"  Conservative endpoint load : {r['conservative_endpoint_power_W']:>10,.3f} W  ({r['conservative_endpoint_power_kW']:.5f} kW)")
    if r["cop"] != 1.0:
        print(f"  Input power (avg)          : {r['input_power_W']:>10,.3f} W  ({r['input_power_kW']:.5f} kW)")
        print(f"  Input power (endpoint)     : {r['input_power_endpoint_W']:>10,.3f} W  ({r['input_power_endpoint_kW']:.5f} kW)")
    print("=" * W)


def plot_temperature_over_time(result, duration_minutes=None, steps=1000):
    t_init = result["initial_temp_c"]
    t_target = result["target_temp_c"]
    t_outer = result["outer_temp_c"]
    dur_s = (duration_minutes * 60) if duration_minutes else result["time_seconds"]
    tau = dur_s / 3.0

    r_in_mm = result["inner_radius_mm"]
    h_in_mm = result["inner_height_mm"]
    mass_g = result["cooled_mass_kg"] * 1000
    u = result["u_coeff_w_m2_c"]
    flow = result["flow_rate_ml_s"]
    cop = result["cop"]

    t_arr = np.linspace(0, dur_s, steps)
    t_min = t_arr / 60
    T = t_target + (t_init - t_target) * np.exp(-t_arr / tau)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.grid(True, alpha=0.3)
    ax.axhline(t_target, linestyle="--", alpha=0.7, label=f"Target  {t_target:.1f} °C")
    ax.axhline(t_outer, linestyle="--", alpha=0.5, label=f"Outer mass  {t_outer:.1f} °C")
    ax.plot(t_min, T, linewidth=2.2, label="Inner zone temperature")
    ax.set_xlim(0, dur_s / 60)
    ax.set_ylim(t_target - 0.4, t_init + 0.4)
    ax.set_xlabel("Time  (minutes)")
    ax.set_ylabel("Temperature  (°C)")
    ax.set_title(
        "Inner Zone Temperature over Time\n"
        f"r = {r_in_mm:.0f} mm  ·  h = {h_in_mm:.0f} mm  ·  "
        f"mass = {mass_g:.0f} g  ·  U = {u:.0f} W/m²·°C  ·  "
        f"flow = {flow:.2f} mL/s  ·  COP = {cop:.1f}",
        fontsize=10,
        pad=10,
    )
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_counterflow_profiles(counterflow_result):
    x = counterflow_result["x_m"]
    Tc = counterflow_result["Tc_c"]
    Th = counterflow_result["Th_c"]
    To = counterflow_result["To_c"]

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(x, Tc, label="Cold inflow")
    ax.plot(x, Th, label="Hot line")
    ax.plot(x, To, label="Warm outflow")
    ax.set_xlabel("Length  (m)")
    ax.set_ylabel("Temperature  (°C)")
    ax.set_title("Counterflow Temperature Profiles")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()


def prompt_u_coeff():
    print("\n  Wall heat-transfer coefficient (U) options:")
    print("    [0] Enter custom value (W/m²·°C)")
    keys = list(U_PRESETS.keys())
    for i, k in enumerate(keys):
        print(f"    [{i+1}] {k:<35} {U_PRESETS[k]:>6} W/m²·°C")
    choice = input("  Select option: ").strip()
    if choice == "0":
        return float(input("  Enter U (W/m²·°C): ").strip())
    try:
        idx = int(choice) - 1
        selected = keys[idx]
        val = U_PRESETS[selected]
        print(f"  -> Using '{selected}': {val} W/m²·°C")
        return float(val)
    except (ValueError, IndexError):
        print("  Invalid - defaulting to tissue conduction (25 W/m²·°C)")
        return 25.0


def run_interactive():
    print("\n" + "=" * 62)
    print("      COOLING POWER + COUNTERFLOW CALCULATOR")
    print("=" * 62)

    try:
        print("\n  -- Inner (cooled) cylinder --")
        r_in = float(input("  Radius (mm)                           : ").strip()) / 1000
        h_in = float(input("  Height (mm)                           : ").strip()) / 1000

        print("\n  -- Outer (heat-producing) cylinder --")
        r_out = float(input("  Radius (mm)                           : ").strip()) / 1000
        h_out = float(input("  Height (mm)                           : ").strip()) / 1000

        print("\n  -- Masses --")
        total = float(input("  Total heat-producing mass (kg)        : ").strip())
        cooled_in = input("  Cooled zone mass (kg) [Enter=auto]    : ").strip()
        if cooled_in:
            cooled = float(cooled_in)
        else:
            vol_L = cylinder_volume(r_in, h_in) * 1000
            cooled = vol_L * DENSITY_WATER_KG_L
            print(f"  -> Auto-derived from volume: {cooled:.4f} kg")

        print("\n  -- Temperatures & time --")
        t_init = float(input("  Initial temperature (C)               : ").strip())
        t_final = float(input("  Target temperature (C)                : ").strip())
        t_outer_in = input("  Outer mass temperature (C) [=initial] : ").strip()
        t_outer = float(t_outer_in) if t_outer_in else t_init
        mins = float(input("  Cooling time (minutes)                : ").strip())

        print("\n  -- Wall heat transfer --")
        u_val = prompt_u_coeff()

        print("\n  -- Axial fluid flow --")
        flow_in = input("  Warm fluid inflow rate (mL/s) [0]     : ").strip()
        flow_ml = float(flow_in) if flow_in else 0.0

        print("\n  -- System --")
        cop_in = input("  System COP [1.0]                      : ").strip()
        cop_val = float(cop_in) if cop_in else 1.0

        print("\n  -- Counterflow line losses --")
        use_cf = input("  Include counterflow line loss? [y/n]  : ").strip().lower()

        counterflow_params = None

        if use_cf in ("y", "yes"):
            T_supply = float(input("  Cold supply temperature (C)                 : ").strip())
            T_out_in = input("  Warm return temperature (C) [Enter=solve]   : ").strip()
            T_out_meas = float(T_out_in) if T_out_in else None
            mdot_cool = float(input("  Coolant flow rate (kg/s)                    : ").strip())
            T_hot_in = float(input("  Hot line distal inlet temp (C)              : ").strip())
            mdot_hot = float(input("  Hot line flow rate (kg/s)                   : ").strip())
            wall_height = float(input("  Shared wall height (m)                      : ").strip())
            wall_length = float(input("  Parallel run length (m)                     : ").strip())
            wall_thick = float(input("  Wall thickness (m)                          : ").strip())
            k_wall = float(input("  Wall conductivity (W/m*K)                   : ").strip())
            h_conv = float(input("  Convective HTC (W/m^2*K)                    : ").strip())

            counterflow_params = {
                "T_supply_c": T_supply,
                "T_out_meas_c": T_out_meas,
                "mdot_cool_kg_s": mdot_cool,
                "cp_cool_j_kgk": water_specific_heat,
                "T_hot_in_c": T_hot_in,
                "mdot_hot_kg_s": mdot_hot,
                "cp_hot_j_kgk": water_specific_heat,
                "wall_height_m": wall_height,
                "wall_length_m": wall_length,
                "wall_thick_m": wall_thick,
                "k_wall_w_mk": k_wall,
                "h_conv_w_m2k": h_conv,
            }

        result = calculate_cooling_power(
            cooled_mass_kg=cooled,
            total_mass_kg=total,
            initial_temp_c=t_init,
            target_temp_c=t_final,
            time_seconds=mins * 60,
            inner_radius_m=r_in,
            inner_height_m=h_in,
            outer_radius_m=r_out,
            outer_height_m=h_out,
            u_coeff_w_m2_c=u_val,
            outer_temp_c=t_outer,
            flow_rate_ml_s=flow_ml,
            cop=cop_val,
            counterflow_params=counterflow_params,
        )

        print()
        print_report(result)
        plot_temperature_over_time(result)

        if result["counterflow_result"] is not None:
            plot_counterflow_profiles(result["counterflow_result"])

        return result

    except Exception as e:
        print(f"\n  Error: {e}")
        return None

: 

In [ ]:
result = run_interactive()

In [ ]:
# References

# Script written by Jane Tunde Kelleher using the follow resources:
# 1) Claude AI (discussion of approach with significant revisions by JTK, assistance with writing & debugging code)
# 2) ChatGPT (discussion of approach with significant revisions by JTK, assistance with writing & debugging code, plot formatting)
# 3) Conversation with Antoine Remy
# 4) Conversation with 2.75 course leadership
# 5) https://en.wikipedia.org/wiki/Counterflow
# 6) https://pmc.ncbi.nlm.nih.gov/articles/PMC8759981/
# 7) https://en.wikipedia.org/wiki/Thermal_transmittance
# 8) https://en.wikipedia.org/wiki/Human_power